In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum, col
import pandas as pd

In [3]:
df = pd.read_csv("data/customers.csv")
df1 = pd.read_csv("data/order_items.csv")
df2 = pd.read_csv("data/orders.csv")
df3 = pd.read_csv("data/products.csv")
df4 = pd.read_csv("data/returns.csv")

In [5]:
totals = {
    "total_customers": df["customer_id"].nunique(),
    "total_products": df3["product_id"].nunique(),
    "total_orders": df2["order_id"].nunique(),
    "total_returned_orders": df4["order_id"].nunique()
}
q1_result = pd.DataFrame([totals])
q1_result.to_csv("output/q1.csv", index=False, header=True)

In [46]:
df.count()

customer_id          100000
customer_name        100000
city                 100000
state                100000
registration_date    100000
customer_segment     100000
dtype: int64

In [47]:
df1.count()

order_item_id    3000000
order_id         3000000
product_id       3000000
quantity         3000000
selling_price    3000000
dtype: int64

In [48]:
df2.count()

order_id        1000000
customer_id     1000000
order_date      1000000
payment_mode    1000000
order_status    1000000
dtype: int64

In [49]:
df3.count()

product_id      50000
product_name    50000
category        50000
brand           50000
unit_cost       50000
dtype: int64

In [50]:
df4.count()

return_id        100000
order_id         100000
return_date      100000
return_reason    100000
dtype: int64

In [51]:
df4.head(10)

,return_id,order_id,return_date,return_reason
0,1,391349,2024-08-06,Changed Mind
1,2,456171,2024-06-28,Arrived Damaged
2,3,977675,2024-09-26,Size Issue
3,4,80452,2024-07-30,Arrived Damaged
4,5,10920,2024-06-15,Size Issue
5,6,365396,2024-08-16,Wrong Item Shipped
6,7,675259,2024-10-02,Arrived Damaged
7,8,871061,2024-11-02,Defective Product
8,9,996307,2024-04-11,Defective Product
9,10,390334,2024-07-06,Wrong Item Shipped


In [ ]:
categoryAmt=products.groupby("category").sum("unit_cost")
categoryAmt.write.csv("output/q2",header=True)

In [9]:
order_customer_items = df2.merge(df1, on="order_id", how="inner")
customer_sales = order_customer_items.merge(df, on="customer_id", how="inner")
customer_sales["purchase_amount"] = customer_sales["quantity"] * customer_sales["selling_price"]
customerAmt = customer_sales.groupby(["customer_id","customer_name"]).sum(numeric_only=True)[["purchase_amount"]]
top10_customers = customerAmt.sort_values("purchase_amount", ascending=False).head(10)
top10_customers.to_csv("output/q3.csv", header=True)

In [12]:
df2["order_date"] = pd.to_datetime(df2["order_date"])
order_sales = df2.merge(df1, on="order_id", how="inner")
order_sales["purchase_amount"] = order_sales["quantity"] * order_sales["selling_price"]
order_sales["year"] = order_sales["order_date"].dt.year
order_sales["month"] = order_sales["order_date"].dt.month
last_year = order_sales["year"].max()
last_year_sales = order_sales[order_sales["year"] == last_year]
monthly_trends = (
    last_year_sales.groupby("month")["purchase_amount"]
    .sum()
    .reset_index()
    .sort_values("month")
)
monthly_trends.to_csv("output/q4.csv", index=False, header=True)

In [7]:
sales_data = df1.merge(df3, on="product_id", how="inner")
category_sales = (
    sales_data.groupby("category")["quantity"]
    .sum()
    .reset_index()
    .rename(columns={"quantity": "total_sold"})
)
returns_data = df4.merge(df1, on="order_id", how="inner").merge(df3, on="product_id", how="inner")
category_returns = (
    returns_data.groupby("category")["quantity"]
    .sum()
    .reset_index()
    .rename(columns={"quantity": "total_returned"})
)
category_stats = category_sales.merge(category_returns, on="category", how="left").fillna(0)
category_stats["return_percentage"] = (
    category_stats["total_returned"] / category_stats["total_sold"] * 100
)
catta = category_stats.sort_values("return_percentage", ascending=False)
catta.to_csv("output/q5.csv", index=False, header=True)

In [10]:
orders_with_state = df2.merge(df, on="customer_id", how="inner")
payment_counts = (
    orders_with_state.groupby(["state", "payment_mode"])
    .size()
    .reset_index(name="count")
)
preferred_payment = (
    payment_counts.loc[
        payment_counts.groupby("state")["count"].idxmax()
    ]
    .reset_index(drop=True)
)
preferred_payment.to_csv("output/q6.csv", index=False, header=True)

In [3]:
customer_orders = (
    df2.merge(df1, on="order_id", how="inner")
       .merge(df3, on="product_id", how="inner")
       .merge(df, on="customer_id", how="inner")
)
customer_orders["purchase_amount"] = customer_orders["quantity"] * customer_orders["selling_price"]
customer_summary = (
    customer_orders.groupby(["customer_id", "customer_name"])
    .agg(
        categories_bought=("category", "nunique"),
        total_spent=("purchase_amount", "sum")
    )
    .reset_index()
)
qualified_customers = customer_summary[
    (customer_summary["categories_bought"] >= 5) &
    (customer_summary["total_spent"] > 100000)
]
atta = qualified_customers.sort_values("total_spent", ascending=False)
atta.to_csv("output/q7.csv", index=False, header=True)

In [4]:
product_sales = df1.merge(df3, on="product_id", how="inner")
product_sales["revenue"] = product_sales["quantity"] * product_sales["selling_price"]
category_product_revenue = (
    product_sales.groupby(["category", "product_name"])["revenue"]
    .sum()
    .reset_index()
)
category_product_revenue = category_product_revenue.sort_values(
    ["category", "revenue"], ascending=[True, False]
)
top3_products_per_category = (
    category_product_revenue.groupby("category").head(3).reset_index(drop=True)
)
top3_products_per_category.to_csv("output/q8.csv", index=False, header=True)